In [ ]:
"""
Brain MRI Analysis for Alzheimer's Detection (AD vs CN)
Uses Random Forest to classify subjects based on brain segmentation metrics.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==========================================
# ⚙️ 설정 (Configuration)
# ==========================================
# 실행할 때 csv 파일 경로를 본인 환경에 맞게 수정하세요.
CSV_PATH = "oasis_brain_analysis_final_200.csv"

def run_analysis(file_path):
    # --------------------------------------
    # 1. 데이터 로드 (Data Loading)
    # --------------------------------------
    if not os.path.exists(file_path):
        print(f"❌ Error: 파일을 찾을 수 없습니다. 경로를 확인하세요: {file_path}")
        return

    df = pd.read_csv(file_path)
    print(f"✅ 데이터 로드 완료: {len(df)} samples")

    # --------------------------------------
    # 2. 전처리 (Preprocessing)
    # --------------------------------------
    # 핵심 로직: MCI 제외하고 AD(치매) vs CN(정상) 이진 분류로 설정
    df_binary = df[df['Group'] != 'MCI'].copy()

    # 학습에 불필요한 메타데이터 제거 (Ratio 데이터가 있다면 절대부피 등은 제거 고려)
    drop_cols = ['Case_ID', 'Group', 'Total_Brain_Voxels']
    X = df_binary.drop(columns=drop_cols, errors='ignore')

    # 타겟 라벨 인코딩 (AD=1, CN=0)
    y = df_binary['Group'].map({'AD': 1, 'CN': 0})

    # 학습/테스트 데이터 분리 (80:20)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"🔹 학습 데이터: {X_train.shape}, 테스트 데이터: {X_test.shape}")

    # --------------------------------------
    # 3. 모델 학습 (Modeling)
    # --------------------------------------
    print("\n🌲 Random Forest 모델 학습 중...")
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # --------------------------------------
    # 4. 평가 (Evaluation)
    # --------------------------------------
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"\n🏆 테스트 정확도(Accuracy): {accuracy * 100:.2f}%")
    print("-" * 50)
    print(classification_report(y_test, y_pred, target_names=['CN (Normal)', 'AD (Dementia)']))
    print("-" * 50)

    # --------------------------------------
    # 5. 시각화 (Visualization)
    # --------------------------------------
    # (1) Confusion Matrix
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['CN', 'AD'], yticklabels=['CN', 'AD'])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')

    # (2) Feature Importance (Top 10)
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:10] # Top 10

    plt.subplot(1, 2, 2)
    plt.title("Top 10 Features Importance")
    plt.barh(range(10), importances[indices], align="center", color='skyblue')
    plt.yticks(range(10), [X.columns[i] for i in indices])
    plt.gca().invert_yaxis()
    plt.xlabel("Importance Score")

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    run_analysis(CSV_PATH)